In [89]:
import sys 
import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy import text

port_mysql = '3306'
user_valentina = "flazo"
pwd_valentina = "T4rg3t2026$$"
server_valentina = "db.mastermold.dev"
db_valentina = "crm_target"
filename_retiro_telef='blacklist_celulares.txt'
fecha_ref='2026-05-01'
tb_name='alfcc_clientes'
periodo='mayo 2026'


def obtener_dni_cliente(tb_name,periodo):
    query = f"""
    SELECT 
    DISTINCT 
    NUMERO_DOCUMENTO as col_01
    FROM {tb_name}
    WHERE cl_base = '{periodo}'
    and cl_estado=1
    """

    df_retiro_dni = pd.read_sql(query, engine_mysql)

    df_retiro_dni["col_01"] = (
        df_retiro_dni["col_01"]
        .astype(str)
        .str.zfill(8)
    )
    return df_retiro_dni

def quitar_telf_enriquecidos(df_retiro_telef,engine_mysql,tb_name):

    if df_retiro_telef.empty:
        print("vacío")
        return
    list_numero = (
        df_retiro_telef['col_02']
        .drop_duplicates()
        .tolist()
    )


    in_clause = ",".join(f"'{x}'" for x in list_numero)
    print(in_clause)

    for col in cols_tel:

        query = f"""
        UPDATE crm_target.{tb_name}
        SET {col} = 0
        WHERE cl_base = 'mayo 2026'
        AND cl_estado = 1
        AND {col} IN ({in_clause})
        """

        with engine_mysql.begin() as conn:
            result = conn.execute(text(query))
            print(f"{col}: {result.rowcount} filas actualizadas")

def tb_retiro_telef(query, engine_mysql,cols_tel,df_list,df_venta):
    df_base_mes = pd.read_sql(query, engine_mysql)

    df = df_base_mes.melt(
        id_vars='col_01',
        value_vars=cols_tel,
        var_name='tipo_telf',
        value_name='col_02'
    )
    df['col_02'] = (
        df['col_02']
        .fillna(0)            
        .astype('int64')        
        .astype(str)              
    )
    df = df[
        (df['col_02'].notna()) &
        (df['col_02'] != '') &
        (df['col_02'] != '0') &
        (df['col_02'].str.len() == 9) &
        (df['col_02'].str.startswith('9'))
    ]


    df_list['col_02'] = (
        df_list['col_02']
        .fillna(0)
        .astype('int64')        
        .astype(str)              
    )

    df = df .merge(
        df_list,
        on="col_02",
        how="inner"
    )

    ids_quitar = df_venta['col_01'].drop_duplicates()

    df= df[
        ~df['col_01'].isin(ids_quitar)
    ]
    return df.drop_duplicates(
        subset=['col_01'],
        keep='last'
    )

def update_base_cliente(engine_mysql,df,tb_name):
    if df.empty:
        print("vacio")
        return
    query = """
    DELETE FROM crm_target.tb_temporal
    """

    with engine_mysql.begin() as conn:
        result = conn.execute(text(query))
        print("Filas eliminadas:", result.rowcount)

    df[['col_01','col_02','col_03']].to_sql(
        name="tb_temporal",
        con=engine_mysql,
        if_exists="append",
        index=False,
        chunksize=1000
    )

    query = f"""
    UPDATE crm_target.{tb_name} a
    INNER JOIN crm_target.tb_temporal b
        ON a.NUMERO_DOCUMENTO = b.col_01 
    SET 
        a.cl_estado = b.col_03,
        a.estado = b.col_02
    WHERE 
        a.cl_base = 'mayo 2026'
    """

    with engine_mysql.begin() as conn:
        result = conn.execute(text(query))
        print("Filas afectadas:", result.rowcount)

def preparar_info(df_list,df_base,df_venta,tipo_retiro):
    if df_list.empty:
        print("vacio")
        return
    df_list['col_01'] = (
        df_list['col_01']
        .fillna(0)            
        .astype('int64')        
    )
    df_list = df_list[df_list['col_01'] != 0].copy()
    df_list['col_01'] = (
        df_list['col_01']
        .astype(str)
        .str.zfill(8)
    )
    df_list = df_list.merge(
        df_base,
        on="col_01",
        how="inner"
    )

    ids_quitar = df_venta['col_01'].drop_duplicates()

    df_list= df_list[
        ~df_list['col_01'].isin(ids_quitar)
    ]
    df_list['col_02']=tipo_retiro
    df_list['col_03']='3'
    print(f"df_list filas: {df_list.shape[0]}")
    return df_list

def venta_periodo(fecha_ref):
    query = f"""
    select distinct dni as col_01 from crm_target.alfcc_ventas
    where fecha>='{fecha_ref}'
    """

    return pd.read_sql(query, engine_mysql)


engine_mysql = create_engine(
    f"mysql+pymysql://{user_valentina}:{pwd_valentina}@{server_valentina}:{port_mysql}/{db_valentina}"
)




In [ ]:

filePath = os.path.join(ruta_csv, filename_retiro_telef)
df_list = pd.read_csv(filePath)

reemplazo = {
    'CELULAR': 'col_02',
}
df_list = df_list.rename(columns=reemplazo)

df_venta=venta_periodo(fecha_ref)

In [ ]:

# evaluar numero enriquecidos

cols_tel = ['cl_telf7', 'cl_telf8', 'cl_telf9', 'cl_telf10']

query_telf_enriquecidos = """
SELECT  NUMERO_DOCUMENTO as col_01, cl_telf7, cl_telf8, cl_telf9, cl_telf10 
FROM crm_target.alfcc_clientes
WHERE cl_base = 'mayo 2026'
and cl_estado=1
"""

df_retiro_telef_enriquecidos=tb_retiro_telef(query_telf_enriquecidos, engine_mysql,cols_tel,df_list,df_venta)
quitar_telf_enriquecidos(df_retiro_telef_enriquecidos,engine_mysql)


'951892894','971779220','910934217','959935183','960841136','981243142','953275043','962361246','986523837','985484839','936918089','938853248','959310306','936308835','943014102','946555791','922690772','990156898','957749276','959359280','979193705','973620677','944375330','969706461','912133622','962530757','981666715','937637376','971310287','947055748','932978902','998226665','952318634','914627600','980734302','964387676','940863418','966808055','922145823','956163993','936651667','950690773','995918182','947004975','968960964','973698121'
cl_telf7: 3 filas actualizadas
cl_telf8: 30 filas actualizadas
cl_telf9: 11 filas actualizadas
cl_telf10: 2 filas actualizadas


In [ ]:

# evaluar numero 

cols_tel = ['cl_telf1', 'cl_telf2', 'cl_telf3', 'cl_telf4','cl_telf5', 'cl_telf6','cl_movil', 'cl_celular', 'cl_telefono']

query_telf = """
SELECT  NUMERO_DOCUMENTO as col_01,cl_telf1, cl_telf2, cl_telf3, cl_telf4, cl_telf5, cl_telf6,cl_movil, cl_celular, cl_telefono 
FROM crm_target.alfcc_clientes
WHERE cl_base = 'mayo 2026'
and cl_estado=1
"""

df_retiro_telef=tb_retiro_telef(query_telf, engine_mysql,cols_tel,df_list,df_venta)

df_retiro_telef['col_02']='Retirar Telf'
df_retiro_telef['col_03']='3'

update_base_cliente(engine_mysql,df)

Filas eliminadas: 1
Filas afectadas: 1


In [88]:

filename='blacklist_dni.txt'
tipo_retiro='Retirar Blacklist'

obtener_dni_cliente(tb_name,periodo)
filePath = os.path.join(ruta_csv, filename)
df_list = pd.read_csv(filePath)
df_list.rename(columns={'DNI': 'col_01'}, inplace=True)

print(df_list.columns.tolist())

['col_01']


In [ ]:
df=preparar_info(df_list,df_retiro_dni,df_venta,tipo_retiro)
update_base_cliente(engine_mysql,df)

df_list filas: 1
Filas eliminadas: 1
Filas afectadas: 1


,col_01,col_02,col_03
0,10218026,Retirar Blacklist,3


In [ ]:
filename='blacklist_dni.txt'
tipo_retiro='Retirar Blacklist'

obtener_dni_cliente(tb_name,periodo)
filePath = os.path.join(ruta_csv, filename)
df_list = pd.read_csv(filePath)
df_list.rename(columns={'DNI': 'col_01'}, inplace=True)

df_list.head(2)

,col_01
0,46654470
1,75469965


In [ ]:
df=preparar_info(df_list,df_retiro_dni,df_venta,tipo_retiro)
update_base_cliente(engine_mysql,df)


In [ ]:
tb_name='alfcc_clientes'
periodo='mayo 2026'
filename='202605_BAE_RETIRO.txt'
tipo_retiro='Retirar BAE'
# evaluar numero 

obtener_dni_cliente(tb_name,periodo)
filePath = os.path.join(ruta_csv, filename)
df_list = pd.read_csv(filePath)
df_list.rename(columns={'DNI': 'col_01'}, inplace=True)

df_list.columns.tolist()

In [ ]:
df=preparar_info(df_list,df_retiro_dni,df_venta,tipo_retiro)
update_base_cliente(engine_mysql,df)

In [83]:
exec_query_sql(server_sa, "VALENTINA", user_sa, pwd_sa, "EXEC VALENTINA.DBO.ALFCC_BASE_VALENTINA", "SP actualizar BASE CREDICASH")


SP actualizar BASE CREDICASH | realizado | duración: 90.36 seg


In [2]:
from sqlalchemy import create_engine


engine_mysql = create_engine(
    f"mysql+pymysql://{user_valentina}:{pwd_valentina}@{server_valentina}:{port_mysql}/{db_valentina}"
)


In [5]:
print(
    df_venta['dni']
    .drop_duplicates()
    .tolist()
)

['25852137', '44426603', '20029022', '40579033', '23967029', '74405803', '23983904', '05232951', '03361702', '45403836', '46774549', '02719548', '27410504', '26683936', '71992333', '60163637', '01049208', '00838704', '41873842', '45095217', '40421730', '73760117', '47037111', '32108912', '45235705', '21504716', '15432337', '15433829', '73576237', '18072514', '16420897', '17948854', '44360032', '43054854', '02601400', '07982417', '45110406', '22506216', '76141639', '09257700', '02386913', '70042019', '07075817', '41954118', '43264418', '47583432', '73270756', '18125310', '29440435', '47056135', '32136518', '19553044', '01137759', '44041454', '04045236', '48189736', '10755149', '45052515', '02835738', '73339751', '41547617', '42453133', '76518732', '16777919', '45308738', '43177900', '44073750', '22311147', '00092431', '76040836', '43899650', '80231853', '45071437', '80158850', '26958644', '00249233', '48367054', '40073657', '43280854', '43705143', '07458148', '03568731', '41986218', '41

In [3]:

ruta_archivo = os.path.join(ruta_csv, 'llamadas_01.xlsx')
df_credicash.to_excel(ruta_archivo, index=False)

ruta_archivo = os.path.join(ruta_csv, 'maquina.xlsx')
df_llamada.to_excel(ruta_archivo, index=False)

### evaluar numero enriquecidos

In [2]:
from sqlalchemy import create_engine


engine_mysql = create_engine(
    f"mysql+pymysql://{user_valentina}:{pwd_valentina}@{server_valentina}:{port_mysql}/{db_valentina}"
)

query = """
SELECT  NUMERO_DOCUMENTO, cl_telf8, cl_telf9, cl_telf10 
FROM crm_target.alfcc_clientes
WHERE cl_base = 'mayo 2026'
and cl_estado=1
"""

df_dni = pd.read_sql(query, engine_mysql)

cols_tel = [
    'cl_telf8','cl_telf9','cl_telf10'
]

df_long = df_dni.melt(
    id_vars='NUMERO_DOCUMENTO',
    value_vars=cols_tel,
    var_name='tipo_telf',
    value_name='CELULAR'
)
df_long['CELULAR'] = (
    df_long['CELULAR']
    .fillna(0)            
    .astype('int64')        
    .astype(str)              
)
df_long = df_long[
    (df_long['CELULAR'].notna()) &
    (df_long['CELULAR'] != '') &
    (df_long['CELULAR'].str.len() == 9) &
    (df_long['CELULAR'].str.startswith('9'))
]



In [3]:
filename='blacklist_celulares.txt'
filePath = os.path.join(ruta_csv, filename)

df_list = pd.read_csv(filePath)
df_list['CELULAR'] = (
    df_list['CELULAR']
    .fillna(0)            
    .astype('int64')        
    .astype(str)              
)
df_list.head()

,CELULAR
0,900000202
1,900000323
2,900000531
3,900000910
4,900001728


In [4]:
df_list['CELULAR'] = df_list['CELULAR'].astype(str)

df_list = df_list.merge(
    df_long,
    on="CELULAR",
    how="inner"
)

print(f"df_list filas: {df_list.shape[0]}")


df_list filas: 4


In [5]:
print(
    df_list['CELULAR']
    .drop_duplicates()
    .tolist()
)

['981296029', '987158466', '987899306', '998986061']


In [ ]:
from sqlalchemy import text
list_numero=['981296029', '987158466', '987899306', '998986061']
query = f"""
update crm_target.alfcc_clientes
set cl_telf8=0
where cl_base='mayo 2026'
and cl_estado=1
and cl_telf8 in(
{list_numero}
)
"""
with engine_mysql.begin() as conn:
    result = conn.execute(text(query))
    print("Filas actualizadas:", result.rowcount)

Filas actualizadas: 11


In [7]:
list_numero=['981296029', '987158466', '987899306', '998986061']


In [8]:
print(list_numero.tolist())

AttributeError: 'list' object has no attribute 'tolist'

In [9]:
from sqlalchemy import text

query = """
DELETE FROM crm_target.tb_temporal
"""

with engine_mysql.begin() as conn:
    result = conn.execute(text(query))
    print("Filas eliminadas:", result.rowcount)

Filas eliminadas: 464


## actualizar retiro telef 

In [ ]:
from sqlalchemy import create_engine


engine_mysql = create_engine(
    f"mysql+pymysql://{user_valentina}:{pwd_valentina}@{server_valentina}:{port_mysql}/{db_valentina}"
)

query = """
SELECT  NUMERO_DOCUMENTO,cl_telf1, cl_telf2, cl_telf3, cl_telf4, cl_telf5, cl_telf6, cl_telf7,cl_movil, cl_celular, cl_telefono 
FROM crm_target.alfcc_clientes
WHERE cl_base = 'mayo 2026'
and cl_estado=1
"""

df_dni = pd.read_sql(query, engine_mysql)

cols_tel = [
    'cl_telf1','cl_telf2','cl_telf3','cl_telf4','cl_telf5',
    'cl_telf6',
    'cl_movil','cl_celular','cl_telefono'
]

df_long = df_dni.melt(
    id_vars='NUMERO_DOCUMENTO',
    value_vars=cols_tel,
    var_name='tipo_telf',
    value_name='CELULAR'
)
df_long['CELULAR'] = (
    df_long['CELULAR']
    .fillna(0)            
    .astype('int64')        
    .astype(str)              
)
df_long = df_long[
    (df_long['CELULAR'].notna()) &
    (df_long['CELULAR'] != '') &
    (df_long['CELULAR'].str.len() == 9) &
    (df_long['CELULAR'].str.startswith('9'))
]



In [3]:
filename='202605_BAE_RETIRO.txt'
filePath = os.path.join(ruta_csv, filename)
df_list = pd.read_csv(filePath)
df_list.head()

,DNI
0,46654470
1,75469965
2,72765590
3,31620060
4,45777360


In [ ]:
filename='blacklist_celulares.txt'
filePath = os.path.join(ruta_csv, filename)

df_list = pd.read_csv(filePath)
df_list['CELULAR'] = (
    df_list['CELULAR']
    .fillna(0)            
    .astype('int64')        
    .astype(str)              
)
df_list.head()

,CELULAR
0,900000202
1,900000323
2,900000531
3,900000910
4,900001728


In [12]:
df_list['CELULAR'] = df_list['CELULAR'].astype(str)

df_list = df_list.merge(
    df_long,
    on="CELULAR",
    how="inner"
)

print(f"df_list filas: {df_list.shape[0]}")


df_list filas: 901


In [13]:
from sqlalchemy import text

query = """
DELETE FROM crm_target.tb_temporal
"""

with engine_mysql.begin() as conn:
    result = conn.execute(text(query))
    print("Filas eliminadas:", result.rowcount)

Filas eliminadas: 0


In [14]:
reemplazo = {
    'NUMERO_DOCUMENTO': 'col_01',
}
df_list = df_list.rename(columns=reemplazo)

df_list['col_02'] ='Retiro Telf'
df_list['col_03'] ='3'
df_list .head()

,CELULAR,col_01,tipo_telf,col_02,col_03
0,900093328,42034808,cl_telf6,Retiro Telf,3
1,900584636,29683346,cl_telf6,Retiro Telf,3
2,900620475,70012357,cl_telf6,Retiro Telf,3
3,901076318,20520227,cl_telf1,Retiro Telf,3
4,901076318,20520227,cl_telefono,Retiro Telf,3


In [15]:
df_list.head()

,CELULAR,col_01,tipo_telf,col_02,col_03
0,900093328,42034808,cl_telf6,Retiro Telf,3
1,900584636,29683346,cl_telf6,Retiro Telf,3
2,900620475,70012357,cl_telf6,Retiro Telf,3
3,901076318,20520227,cl_telf1,Retiro Telf,3
4,901076318,20520227,cl_telefono,Retiro Telf,3


In [16]:
df_list = df_list.drop_duplicates(
    subset=['col_01'],
    keep='last'
)
df_list.count()

CELULAR      684
col_01       684
tipo_telf    684
col_02       684
col_03       684
dtype: int64

In [17]:

df_list["col_01"] = (
    df_list["col_01"]
    .astype(str)
    .str.zfill(8)
)


In [18]:

df_list[['col_01','col_02','col_03']].to_sql(
    name="tb_temporal",
    con=engine_mysql,
    if_exists="append",
    index=False,
    chunksize=1000
)

684

In [ ]:
df_list['CELULAR'] 


df_list.to_sql(
    name="tb_temporal",
    con=engine_mysql,
    if_exists="append",
    index=False,
    chunksize=1000
)

,CELULAR
0,900000202
1,900000531
2,900001728
3,900003782
4,900004400


In [11]:
print(
    df_list['CELULAR']
    .drop_duplicates()
    .tolist()
)

['901914305', '902876236', '902893436', '906573460', '912676290', '913300840', '914050257', '914647039', '920033871', '920233049', '920309404', '920569770', '920574786', '920637180', '921729655', '922804891', '923802595', '924261113', '924470592', '926579945', '926931724', '926974122', '927687059', '927910705', '927987418', '928919344', '931740138', '932130621', '932813141', '932941966', '934573818', '935339850', '936052202', '936447502', '936898041', '937173655', '937202437', '937518944', '938132787', '938171773', '939114190', '939173337', '939187202', '939317200', '939373234', '939586724', '939667425', '940781497', '941260823', '941265695', '941276327', '941288702', '941339959', '941467158', '941792920', '941824681', '941897091', '941909576', '942199388', '942993339', '942997731', '943082431', '943425401', '944128135', '944181154', '944415166', '944625644', '944641740', '944726273', '944752465', '945086183', '945173393', '945363397', '945451818', '945552413', '945750143', '945769393'

In [ ]:
df_list['CELULAR'] 

reemplazo = {
    'CELULAR': 'col_02',
}
df_list = df_list.rename(columns=reemplazo)
df_list.to_sql(
    name="tb_temporal",
    con=engine_mysql,
    if_exists="append",
    index=False,
    chunksize=1000
)

PendingRollbackError: Can't reconnect until invalid transaction is rolled back.  Please rollback() fully before proceeding (Background on this error at: https://sqlalche.me/e/20/8s2b)

In [4]:

print(f"df_dni filas: {df_long.shape[0]}")
print(f"df_list filas: {df_list.shape[0]}")
df_list['CELULAR'] = (
    df_list['CELULAR']
    .fillna(0)            
    .astype('int64')        
    .astype(str)              
)

df_dni filas: 190898
df_list filas: 614450


In [26]:
dnis = [
    "70663492",
    "47002810",
    "75797316",
    "61273533",
    "10182850",
    "70884441",
    "07876887"
]

telefonos = [
    "983849201",
    "900927220",
    "986506300",
    "981422534",
    "989341699",
    "941800134",
    "998183675"
]

# import pandas as pd

df_manual_dni = pd.DataFrame({
    "dni_cliente": dnis,
})


df_manual_telf = pd.DataFrame({
    "CELULAR": telefonos
})


In [23]:
df_manual_telf.head()

,CELULAR
0,983849201
1,900927220
2,986506300
3,981422534
4,989341699


In [ ]:
data = [
    ("70663492", "983849201"),
    ("47002810", "900927220"),
    ("75797316", "986506300"),
    ("61273533", "981422534"),
    ("10182850", "989341699"),
    ("70884441", "941800134"),
    ("07876887", "998183675")
]


df_manual = pd.DataFrame(data, columns=["DNI", "CELULAR"])

In [21]:
spark_df = spark.createDataFrame(df_manual)
spark_df.show()


NameError: name 'spark' is not defined

In [25]:
df_manual.head()

,DNI,TELEFONO
0,70663492,983849201
1,47002810,900927220
2,75797316,986506300
3,61273533,981422534
4,10182850,989341699


In [27]:
df_manual_telf['CELULAR'] = df_manual_telf['CELULAR'].astype(str)

df_manual_telf = df_manual_telf.merge(
    df_long,
    on="CELULAR",
    how="inner"
)

print(f"df_list filas: {df_manual_telf.shape[0]}")


df_list filas: 0


In [6]:
df_list['cl_estado']='3'
df_list['estado']='Retirar Telf'
df_list=df_list[['NUMERO_DOCUMENTO','cl_estado','estado']]
df_list.count()

NUMERO_DOCUMENTO    417
cl_estado           417
estado              417
dtype: int64

In [7]:
df_list = df_list.drop_duplicates(
    subset=['NUMERO_DOCUMENTO'],
    keep='last'
)
df_list.count()

NUMERO_DOCUMENTO    216
cl_estado           216
estado              216
dtype: int64

In [10]:
df_list.head()

,col_01,col_03,col_02
1,45773218,3,Retirar Telf
3,43888745,3,Retirar Telf
5,04647692,3,Retirar Telf
7,47240319,3,Retirar Telf
9,71563325,3,Retirar Telf


In [9]:
reemplazo = {
    'NUMERO_DOCUMENTO': 'col_01',
    'estado': 'col_02',
    'cl_estado': 'col_03'
}
df_list = df_list.rename(columns=reemplazo)

In [10]:
from sqlalchemy import create_engine

engine = create_engine(
    f"mysql+pymysql://{user_valentina}:{pwd_valentina}@{server_valentina}:{port_mysql}/{db_valentina}"
)


In [ ]:

df_list.to_sql(
    name="tb_temporal",
    con=engine_mysql,
    if_exists="append",
    index=False,
    chunksize=1000
)

216

In [19]:
from sqlalchemy import text

query = """
UPDATE crm_target.alfcc_clientes a
INNER JOIN crm_target.tb_temporal b
    ON a.NUMERO_DOCUMENTO = b.col_01 
SET 
    a.cl_estado = b.col_03,
    a.estado = b.col_02
WHERE 
    a.cl_base = 'mayo 2026'
"""

with engine_mysql.begin() as conn:
    result = conn.execute(text(query))
    print("Filas afectadas:", result.rowcount)

Filas afectadas: 684


In [20]:
from sqlalchemy import text

query = """
DELETE FROM crm_target.tb_temporal
"""

with engine_mysql.begin() as conn:
    result = conn.execute(text(query))
    print("Filas eliminadas:", result.rowcount)

Filas eliminadas: 70


In [ ]:
from sqlalchemy import text

query = """
UPDATE crm_target.alfcc_clientes a
INNER JOIN crm_target.tb_temporal b
    ON a.NUMERO_DOCUMENTO COLLATE utf8mb4_general_ci
       = b.col_01 COLLATE utf8mb4_general_ci
SET 
    a.cl_estado = b.col_02,
    a.estado = b.col_03
WHERE 
    a.cl_base = 'mayo 2026'
    AND (
        IFNULL(a.cl_estado,'') COLLATE utf8mb4_general_ci 
            <> IFNULL(b.col_02,'') COLLATE utf8mb4_general_ci
        OR 
        IFNULL(a.estado,'') COLLATE utf8mb4_general_ci 
            <> IFNULL(b.col_03,'') COLLATE utf8mb4_general_ci
    )
LIMIT 1000
"""

with engine.begin() as conn:
    total = 0

    while True:
        result = conn.execute(text(query))
        filas = result.rowcount
        total += filas

        print("Filas afectadas:", filas)

        if filas == 0:
            break

    print("TOTAL ACTUALIZADO:", total)

In [ ]:

engine_mysql = create_engine(
    f"mysql+pymysql://{user_valentina}:{pwd_valentina}@{server_valentina}:{port_mysql}/{db_valentina}"
)

query = """
SELECT 
* FROM tb_temporal
"""

df_prueba_01 = pd.read_sql(query, engine_mysql)

df_prueba_01.columns

Index(['col_01', 'col_02', 'col_03', 'col_04'], dtype='object')

In [8]:
update_mysql_en_bloques(
    df=df_list,
    tabla="alfcc_clientes",
    periodo="mayo 2026",
    col_llave_mysql="NUMERO_DOCUMENTO",
    col_valor_mysql="cl_estado",
    col_llave_df="NUMERO_DOCUMENTO",
    col_valor_df="retiro",
    host=server_valentina,
    user=user_valentina,
    password=pwd_valentina,
    database=db_valentina,
    port=port_mysql,
    batch_size=2000,
    validar_sin_grabar=False
)


Total registros a procesar: 11788
Lote 0 - 2000 actualizado | filas afectadas: 1927
Lote 2000 - 4000 actualizado | filas afectadas: 1931
Lote 4000 - 6000 actualizado | filas afectadas: 1908
Lote 6000 - 8000 actualizado | filas afectadas: 1898
Lote 8000 - 10000 actualizado | filas afectadas: 1897
Lote 10000 - 11788 actualizado | filas afectadas: 1690
Proceso terminado. Total filas afectadas: 11251


## actualizar retiro dni

In [5]:
from sqlalchemy import create_engine


engine_mysql = create_engine(
    f"mysql+pymysql://{user_valentina}:{pwd_valentina}@{server_valentina}:{port_mysql}/{db_valentina}"
)

query = """
SELECT 
DISTINCT 
NUMERO_DOCUMENTO as dni_cliente
FROM alfcc_clientes
WHERE cl_base = 'mayo 2026'
and cl_estado=1
"""

df_dni = pd.read_sql(query, engine_mysql)

df_dni["dni_cliente"] = (
    df_dni["dni_cliente"]
    .astype(str)
    .str.zfill(8)
)


In [21]:
filename='blacklist_dni.txt'

filePath = os.path.join(ruta_csv, filename)
df_list = pd.read_csv(filePath)

In [ ]:
filename='blacklist_dni.txt'

filePath = os.path.join(ruta_csv, filename)
df_list = pd.read_csv(filePath)

df_list.rename(columns={'DNI': 'dni_cliente'}, inplace=True)
df_list["dni_cliente"] = (
    df_list["dni_cliente"]
    .astype(str)
    .str.zfill(8)
)
print(df_list.columns)
print(df_dni.columns)


Index(['dni_cliente'], dtype='object')
Index(['dni_cliente'], dtype='object')


In [35]:
print(df_dni.columns)
print(df_list.columns)

Index(['dni_cliente'], dtype='object')
Index(['dni_cliente'], dtype='object')


In [ ]:

exec_query_sql(server_sa, "VALENTINA", user_sa, pwd_sa, "EXEC VALENTINA.DBO.ALFCC_BASE_VALENTINA", "SP actualizar BASE CREDICASH")
exec_query_sql(server_sa, "VALENTINA", user_sa, pwd_sa, "EXEC VALENTINA.DBO.ALFCC_BASE_VALENTINA", "SP actualizar BASE CREDICASH")
exec_query_sql(server_sa, "VALENTINA", user_sa, pwd_sa, "EXEC VALENTINA.DBO.ALFCC_BASE_VALENTINA", "SP actualizar BASE CREDICASH")
exec_query_sql(server_sa, "VALENTINA", user_sa, pwd_sa, "EXEC VALENTINA.DBO.ALFCC_BASE_VALENTINA", "SP actualizar BASE CREDICASH")
exec_query_sql(server_sa, "VALENTINA", user_sa, pwd_sa, "EXEC VALENTINA.DBO.ALFCC_BASE_VALENTINA", "SP actualizar BASE CREDICASH")


SP actualizar BASE CREDICASH | realizado | duración: 91.97 seg


In [7]:
df_list = df_list.merge(
    df_dni,
    on="dni_cliente",
    how="inner"
)

print(f"df_list filas: {df_list.shape[0]}")


df_list filas: 6183


In [8]:
df_list['col_03']=3
df_list['col_02']='Retiro Bae'

In [10]:
reemplazo = {
    'dni_cliente': 'col_01',
    # 'retiro': 'col_02',
}
df_list = df_list.rename(columns=reemplazo)
df_list.head()

,col_01,col_03,col_02
0,73694914,3,Retiro Bae
1,30677717,3,Retiro Bae
2,09639416,3,Retiro Bae
3,41419547,3,Retiro Bae
4,41551915,3,Retiro Bae


In [24]:
df_list['retiro']='Retirar BlackList'
df_list=df_list[['dni_cliente','retiro']]
df_list.count()

dni_cliente    1
retiro         1
dtype: int64

In [12]:
from sqlalchemy import text

query = """
UPDATE crm_target.alfcc_clientes a
INNER JOIN crm_target.tb_temporal b
    ON a.NUMERO_DOCUMENTO = b.col_01 
SET 
    a.cl_estado = b.col_03,
    a.estado = b.col_02
WHERE 
    a.cl_base = 'mayo 2026'
"""

with engine_mysql.begin() as conn:
    result = conn.execute(text(query))
    print("Filas afectadas:", result.rowcount)

Filas afectadas: 6867


In [11]:

df_list[['col_01', 'col_02', 'col_03']].to_sql(
    name="tb_temporal",
    con=engine_mysql,
    if_exists="append",
    index=False,
    chunksize=1000
)

6183

In [ ]:
df_list

In [13]:

ruta_archivo = os.path.join(ruta_csv, 'retiro_credicash.xlsx')
df_list.to_excel(ruta_archivo, index=False)

In [ ]:
update_mysql_en_bloques(
    df=df_list,
    tabla="alfin_clientes",
    periodo="mayo 2026",
    col_llave_mysql="NUMERO_DOCUMENTO",
    col_valor_mysql="estado",
    col_llave_df="dni_cliente",
    col_valor_df="retiro",
    host=server_valentina,
    user=user_valentina,
    password=pwd_valentina,
    database=db_valentina,
    port=port_mysql,
    batch_size=2000,
    validar_sin_grabar=False
)


Total registros a procesar: 1648
Lote 0 - 1648 actualizado | filas afectadas: 0
Proceso terminado. Total filas afectadas: 0


## ejecutar query

In [ ]:
from sqlalchemy import create_engine


engine = create_engine(
    f"mysql+pymysql://{user_valentina}:{pwd_valentina}@{server_valentina}:{port_mysql}/{db_valentina}"
)
with engine.begin() as conn:
    result = conn.execute(text("""
        UPDATE crm_target.alfcc_clientes
        SET cl_estado = 3
        WHERE estado <> 'ACTIVO'
        AND cl_base = 'mayo 2026'
    """))
    
    print("Filas afectadas:", result.rowcount)

## actualizar dni a otro lote

In [3]:
filename='dni_repetidos_alfin.csv'

filePath = os.path.join(ruta_csv, filename)
df_list = pd.read_csv(filePath)

In [5]:
df_list["NUMERO_DOCUMENTO"] = (
    df_list["NUMERO_DOCUMENTO"]
    .astype(str)
    .str.zfill(8)
)
df_list.head()

,NUMERO_DOCUMENTO
0,46507674
1,40963267
2,41782588
3,40658062
4,09457398


In [6]:
df_list['lote_ref']='BD-Target -ASM'

In [7]:
update_mysql_en_bloques(
    df=df_list,
    tabla="alfin_clientes",
    periodo="mayo 2026",
    col_llave_mysql="NUMERO_DOCUMENTO",
    col_valor_mysql="lote",
    col_llave_df="NUMERO_DOCUMENTO",
    col_valor_df="lote_ref",
    host=server_valentina,
    user=user_valentina,
    password=pwd_valentina,
    database=db_valentina,
    port=port_mysql,
    batch_size=2000,
    validar_sin_grabar=False
)


Total registros a procesar: 1066
Lote 0 - 1066 actualizado | filas afectadas: 1066
Proceso terminado. Total filas afectadas: 1066


In [ ]:
BD-Target -ASM

In [5]:
exec_query_sql(server_zeus, "MAEBA", user_zeus, pwd_zeus, "ADM_OBJ_TG.spFunnelDinersTc", "SP funnel diners_tc Zeus")

SP funnel diners_tc Zeus | realizado | duración: 19.96 seg


In [ ]:
overwrite_table_SQL(spark,df_prueba_1,f'borrar_TARGET_202604_01',server_kishin,user_kishin,pwd_kishin,'DANTALION')
overwrite_table_SQL(spark,df_prueba_2,f'borrar_TARGET_202604_02',server_kishin,user_kishin,pwd_kishin,'DANTALION')
overwrite_table_SQL(spark,df_prueba_ch,f'borrar_TARGET_202604_ch',server_kishin,user_kishin,pwd_kishin,'DANTALION')


df_list filas: 1066


In [ ]:
df_dni filas: 128183
df_list filas: 12546

In [ ]:
ssss

In [7]:
print(df_list.columns)
print(df_long.columns)

Index(['TELEFONO'], dtype='object')
Index(['NUMERO_DOCUMENTO', 'tipo_telf', 'TELEFONO'], dtype='object')


df_list filas: 1


In [ ]:
df_list filas: 2036

NUMERO_DOCUMENTO    1
retiro              1
dtype: int64

In [10]:
df_list.head()

,NUMERO_DOCUMENTO,retiro
0,40503275,Retirar Telef


Total registros a procesar: 1
Lote 0 - 1 actualizado | filas afectadas: 1
Proceso terminado. Total filas afectadas: 1
